In [6]:
"""Train the PCA detector and scale its anomaly scores."""

import os
import sys

# Clear any cached modules
if 'config' in sys.modules:
    del sys.modules['config']
if 'to_tflite' in sys.modules:
    del sys.modules['to_tflite']

import pandas as pd
from pyod.models.pca import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import config
import to_tflite

# Build the detector from its configuration.
pca = PCA(
    contamination=config.CONTAMINATION,
    n_components=config.N_COMPONENTS,
    whiten=config.WHITEN,
    weighted=config.WEIGHTED,
    standardization=config.STANDARDIZATION,
    random_state=config.RANDOM_STATE,
)

# NOTE: all dataset features must be pre-normalized so that, after the int8
# quantization in keras/to_tflite.py, no value overflows the int8 range.
# Fit the configured features from the CSV.
features = pd.read_csv(config.CSV_PATH)[config.FEATURES].to_numpy()
X_train, X_test = train_test_split(
    features, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE
)
pca.fit(X_train)

# Scale the raw anomaly scores into the configured range.
raw_scores = pca.decision_function(X_train).reshape(-1, 1)
score_scaler = MinMaxScaler(feature_range=config.SCORE_RANGE).fit(raw_scores)
train_scores = score_scaler.transform(raw_scores).ravel()
test_scores = score_scaler.transform(
    pca.decision_function(X_test).reshape(-1, 1)
).ravel()

# Learn the scores with a small network and export .keras + .tflite.
to_tflite.to_tflite(
    name="pca",
    x_train=X_train,
    y_train=train_scores,
    x_test=X_test,
    y_test=test_scores,
    layers=config.LAYERS,
    learning_rate=config.LEARNING_RATE,
    epochs=config.EPOCHS,
    batch_size=config.BATCH_SIZE,
    verbose=config.VERBOSE,
    feature_names=config.FEATURES,
)


Epoch 1/50


2026-08-14 18:07:51.535858: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 779us/step - loss: 0.0026 - val_loss: 9.9430e-05
Epoch 2/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 770us/step - loss: 8.8698e-05 - val_loss: 8.3628e-05
Epoch 3/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 758us/step - loss: 5.9904e-05 - val_loss: 4.1061e-05
Epoch 4/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 775us/step - loss: 3.4956e-05 - val_loss: 2.3801e-05
Epoch 5/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 759us/step - loss: 2.4942e-05 - val_loss: 1.9560e-05
Epoch 6/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 774us/step - loss: 2.0280e-05 - val_loss: 3.9795e-05
Epoch 7/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 772us/step - loss: 1.8734e-05 - val_loss: 1.4637e-05
Epoch 8/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 764us/step - loss: 1.8332e-05 - val_loss: 1.7669e-05
Epoch 9/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 761us/step - loss: 1.7638e-05 - val_loss: 1.9259e-05
Epoch 10/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 767us/step - loss: 1.7629e-05 - val_loss: 1.4095e-05
Epoch 11/50
3747/3747 ━━━━━━━━━━━━

INFO:tensorflow:Assets written to: /tmp/tmpo9lo8xoq/assets


Saved artifact at '/tmp/tmpo9lo8xoq'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 2), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  140089465070096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140089465073168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140089465071824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140089465069712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140089465073744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140089465070864: TensorSpec(shape=(), dtype=tf.resource, name=None)


/home/patrik/Diplomka/LODA/env/lib/python3.12/site-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1786723819.546711   15783 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1786723819.546723   15783 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2026-08-14 18:10:19.546900: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpo9lo8xoq
2026-08-14 18:10:19.547224: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-08-14 18:10:19.547228: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpo9lo8xoq
I0000 00:00:1786723819.549397   15783 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
2026-08-14 18:10:19.549679: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-08-14 1

<Sequential name=sequential, built=True>